# DQN Lunar Lander — Colab + Gradio

Train **LunarLander-v3** with Stable Baselines3 DQN.

- **Colab**: Edit hyperparams in cells, run cells → learning curve → video. **GPU/CPU supported** (CUDA if GPU runtime, runs on CPU too).
- **Gradio**: Run the Gradio section cell to tune via sliders and train (Colab/local).
- **Experience replay** and **target network** included. After training, save video with **RecordVideo** and play.

## 1. Setup (run first in Colab)

- **Device**: GPU/CPU supported. (Optional) **Runtime → Change runtime type → GPU** for CUDA; runs on CPU too.
- Colab has no display; we save video with `moviepy` then play. Gradio is installed too.

In [ ]:
!pip install stable-baselines3[extra] "gymnasium[classic-control,box2d]" moviepy gradio -q

## 2. Imports and callback

Define a callback that collects episode rewards (for live learning curve).

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*pkg_resources is deprecated.*")

import gymnasium as gym
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from stable_baselines3 import DQN
from stable_baselines3.common.callbacks import BaseCallback


def get_device():
    """GPU and CPU supported: CUDA > MPS > CPU. Uses CUDA if Colab GPU runtime, else CPU."""
    import torch
    if torch.cuda.is_available():
        return "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def get_device_name(device):
    if device is None:
        return "auto"
    s = str(device)
    if "cuda" in s.lower():
        return "CUDA (GPU)"
    if "mps" in s.lower():
        return "Apple Silicon (MPS)"
    return "CPU"


class EpisodeRewardCallback(BaseCallback):
    """Append episode reward to list on each episode end (for live chart)."""
    def __init__(self, reward_list, step_list, verbose=0):
        super().__init__(verbose)
        self.reward_list = reward_list
        self.step_list = step_list

    def _on_rollout_end(self):
        if self.model.ep_info_buffer:
            for info in self.model.ep_info_buffer:
                self.reward_list.append(info["r"])
                self.step_list.append(self.num_timesteps)
        return True

## 3. Hyperparameters

Grouped by category. Change values then run the training cell below.

In [ ]:
# Training scale
TOTAL_TIMESTEPS = 200_000
CHUNK_STEPS = 10_000   # Refresh chart every this many steps
LOG_INTERVAL = 4

# Experience replay
BUFFER_SIZE = 50_000
LEARNING_STARTS = 1_000
BATCH_SIZE = 32
TRAIN_FREQ = 4

# Target network
TARGET_UPDATE_INTERVAL = 1_000
TAU = 1.0

# General
LEARNING_RATE = 1e-3
GAMMA = 0.99
GRADIENT_STEPS = 1

# Exploration
EXPLORATION_FRACTION = 0.2
EXPLORATION_INITIAL_EPS = 1.0
EXPLORATION_FINAL_EPS = 0.05

# Save
SAVE_PATH = "dqn_lunarlander_colab"
VIDEO_FOLDER = "./video"

## 4. Training (chunked + live learning curve)

Clear output and redraw chart each chunk for a live feel.

In [ ]:
from IPython.display import clear_output

device = get_device()
print(f"Device: {get_device_name(device)} (CUDA if GPU, else CPU)")

env = gym.make("LunarLander-v3")
reward_list = []
step_list = []
callback = EpisodeRewardCallback(reward_list, step_list)

model = DQN(
    policy="MlpPolicy",
    env=env,
    device=device,
    learning_rate=LEARNING_RATE,
    buffer_size=BUFFER_SIZE,
    learning_starts=LEARNING_STARTS,
    batch_size=BATCH_SIZE,
    tau=TAU,
    gamma=GAMMA,
    target_update_interval=TARGET_UPDATE_INTERVAL,
    train_freq=TRAIN_FREQ,
    gradient_steps=GRADIENT_STEPS,
    exploration_fraction=EXPLORATION_FRACTION,
    exploration_initial_eps=EXPLORATION_INITIAL_EPS,
    exploration_final_eps=EXPLORATION_FINAL_EPS,
    verbose=1,
)

total = TOTAL_TIMESTEPS
chunk = CHUNK_STEPS

for i in range(0, total, chunk):
    steps_this = min(chunk, total - i)
    model.learn(total_timesteps=steps_this, callback=callback, log_interval=LOG_INTERVAL)

    clear_output(wait=True)
    if reward_list:
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(step_list, reward_list, alpha=0.4, color="steelblue", label="Episode reward")
        if len(reward_list) >= 10:
            w = min(50, len(reward_list) // 5)
            kernel = np.ones(w) / w
            smoothed = np.convolve(reward_list, kernel, mode="same")
            ax.plot(step_list, smoothed, color="coral", linewidth=2, label=f"Moving avg (w={w})")
        ax.set_xlabel("Step")
        ax.set_ylabel("Episode reward")
        ax.set_title(f"Learning curve — {i + steps_this:,} / {total:,} steps")
        ax.legend(loc="lower right")
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    print(f"Progress: {i + steps_this:,} / {total:,} steps, episodes: {len(reward_list)}")

env.close()
print("Training done.")

## 5. Model save and stats

In [ ]:
model.save(SAVE_PATH)
print(f"Model saved: {SAVE_PATH}.zip")

if reward_list:
    last_n = min(100, len(reward_list))
    mean_last = sum(reward_list[-last_n:]) / last_n
    print(f"Last {last_n} episode mean reward: {mean_last:.1f}")
    print(f"Total episodes: {len(reward_list)}")

## 6. Visualization: record and play video (Colab)

Colab has no display for `env.render()`; use **RecordVideo** wrapper to save `.mp4`, then play with `IPython.display.Video`.

In [ ]:
import os
from gymnasium.wrappers import RecordVideo
from IPython.display import Video

# Env for video (render_mode="rgb_array" required)
eval_env = gym.make("LunarLander-v3", render_mode="rgb_array")
trigger = lambda t: t % 1 == 0  # Record every episode
eval_env = RecordVideo(
    eval_env,
    video_folder=VIDEO_FOLDER,
    episode_trigger=trigger,
    disable_logger=True,
)

# Run 1 episode with trained model and record
obs, info = eval_env.reset()
terminated, truncated = False, False
total_reward = 0
while not (terminated or truncated):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = eval_env.step(action)
    total_reward += reward

eval_env.close()
print(f"Episode return: {total_reward:.1f}")

In [ ]:
# Play video in Colab
video_files = [f for f in os.listdir(VIDEO_FOLDER) if f.endswith(".mp4")]
if video_files:
    video_path = os.path.join(VIDEO_FOLDER, sorted(video_files)[-1])
    display(Video(video_path, embed=True))
else:
    print("No video file. Run the record cell above first.")

---

### Summary (notebook style)

1. **Setup**: `stable-baselines3[extra]`, `gymnasium[classic-control,box2d]`, `moviepy`, `gradio`
2. **Colab device**: GPU/CPU supported. GPU → CUDA, CPU-only works too
3. **Training**: DQN + experience replay + target network; learning curve updated each chunk
4. **Visualization**: `RecordVideo` → save `.mp4` → `Video()` play. Run section below for **Gradio**

---

## Gradio app (tune hyperparams with sliders)

Run the cell below to launch Gradio. In Colab the app is embedded in the cell output. **GPU → CUDA, else CPU** automatically. Change sliders then click **Start training**.

In [ ]:
import os
import gradio as gr
from gymnasium.wrappers import RecordVideo

VIDEO_FOLDER_GRADIO = "./video_gradio"
os.makedirs(VIDEO_FOLDER_GRADIO, exist_ok=True)


def run_training_gradio(
    total_timesteps, chunk_steps, log_interval,
    buffer_size, learning_starts, batch_size, train_freq,
    target_update_interval, tau, learning_rate, gamma, gradient_steps,
    exploration_fraction, exploration_initial_eps, exploration_final_eps,
    verbose, save_after, save_path, record_video,
    progress=gr.Progress(),
):
    device = get_device()
    env = gym.make("LunarLander-v3")
    reward_list = []
    step_list = []
    callback = EpisodeRewardCallback(reward_list, step_list)

    model = DQN(
        policy="MlpPolicy",
        env=env,
        device=device,
        learning_rate=float(learning_rate),
        buffer_size=int(buffer_size),
        learning_starts=int(learning_starts),
        batch_size=int(batch_size),
        tau=float(tau),
        gamma=float(gamma),
        target_update_interval=int(target_update_interval),
        train_freq=int(train_freq),
        gradient_steps=int(gradient_steps),
        exploration_fraction=float(exploration_fraction),
        exploration_initial_eps=float(exploration_initial_eps),
        exploration_final_eps=float(exploration_final_eps),
        verbose=1 if verbose else 0,
    )
    total = int(total_timesteps)
    chunk = int(chunk_steps)
    n_chunks = max(1, (total + chunk - 1) // chunk)
    for idx, i in enumerate(range(0, total, chunk)):
        steps_this = min(chunk, total - i)
        model.learn(total_timesteps=steps_this, callback=callback, log_interval=int(log_interval))
        progress((idx + 1) / n_chunks, desc=f"Training... {i + steps_this:,} / {total:,} steps")
    env.close()

    fig = None
    if reward_list:
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(step_list, reward_list, alpha=0.4, color="steelblue", label="Episode reward")
        if len(reward_list) >= 10:
            w = min(50, len(reward_list) // 5)
            smoothed = np.convolve(reward_list, np.ones(w) / w, mode="same")
            ax.plot(step_list, smoothed, color="coral", linewidth=2, label=f"Moving avg (w={w})")
        ax.set_xlabel("Step")
        ax.set_ylabel("Episode reward")
        ax.set_title("Learning curve")
        ax.legend(loc="lower right")
        ax.grid(True, alpha=0.3)
        plt.tight_layout()

    stats_lines = [f"Device: {get_device_name(device)}", f"Total episodes: {len(reward_list)}"]
    if reward_list:
        last_n = min(100, len(reward_list))
        stats_lines.append(f"Last {last_n} episode mean reward: {sum(reward_list[-last_n:]) / last_n:.1f}")
    if save_after and save_path and save_path.strip():
        model.save(save_path.strip())
        stats_lines.append(f"Model saved: {save_path.strip()}.zip")
    stats_text = "\n".join(stats_lines)

    video_path = None
    if record_video:
        eval_env = gym.make("LunarLander-v3", render_mode="rgb_array")
        eval_env = RecordVideo(eval_env, video_folder=VIDEO_FOLDER_GRADIO, episode_trigger=lambda t: t == 0, disable_logger=True)
        obs, _ = eval_env.reset()
        term, trunc = False, False
        while not (term or trunc):
            action, _ = model.predict(obs, deterministic=True)
            obs, _, term, trunc, _ = eval_env.step(action)
        eval_env.close()
        videos = [f for f in os.listdir(VIDEO_FOLDER_GRADIO) if f.endswith(".mp4")]
        if videos:
            video_path = os.path.join(VIDEO_FOLDER_GRADIO, sorted(videos)[-1])
    return fig, stats_text, video_path


with gr.Blocks(title="DQN Lunar Lander", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# DQN Lunar Lander (Gradio)")
    gr.Markdown("Tune hyperparams with sliders · GPU/CPU auto (CUDA if GPU)")
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Training scale")
            total_timesteps = gr.Slider(10_000, 500_000, value=200_000, step=10_000, label="Total timesteps")
            chunk_steps = gr.Slider(2_000, 20_000, value=5_000, step=1_000, label="Chunk steps")
            log_interval = gr.Slider(1, 20, value=4, step=1, label="Log interval")
            gr.Markdown("### Experience replay")
            buffer_size = gr.Slider(5_000, 200_000, value=50_000, step=5_000, label="buffer_size")
            learning_starts = gr.Slider(500, 10_000, value=1_000, step=500, label="learning_starts")
            batch_size = gr.Slider(16, 256, value=32, step=16, label="batch_size")
            train_freq = gr.Slider(1, 32, value=4, step=1, label="train_freq")
            gr.Markdown("### Target network")
            target_update_interval = gr.Slider(100, 5_000, value=1_000, step=100, label="target_update_interval")
            tau = gr.Slider(0.0, 1.0, value=1.0, step=0.05, label="tau")
            gr.Markdown("### General")
            learning_rate = gr.Dropdown(choices=[1e-4, 5e-4, 1e-3, 2e-3, 4e-3, 1e-2], value=1e-3, label="learning_rate")
            gamma = gr.Slider(0.9, 1.0, value=0.99, step=0.01, label="gamma")
            gradient_steps = gr.Slider(1, 16, value=1, step=1, label="gradient_steps")
            gr.Markdown("### Exploration")
            exploration_fraction = gr.Slider(0.05, 0.5, value=0.2, step=0.05, label="exploration_fraction")
            exploration_initial_eps = gr.Slider(0.5, 1.0, value=1.0, step=0.05, label="exploration_initial_eps")
            exploration_final_eps = gr.Slider(0.0, 0.2, value=0.05, step=0.01, label="exploration_final_eps")
            gr.Markdown("### Other")
            verbose = gr.Checkbox(value=True, label="verbose")
            save_after = gr.Checkbox(value=True, label="Save model after training")
            save_path = gr.Textbox(value="dqn_lunarlander_gradio", label="Save path")
            record_video = gr.Checkbox(value=True, label="Record video")
            run_btn = gr.Button("Start training", variant="primary")
        with gr.Column(scale=2):
            plot_out = gr.Plot(label="Learning curve")
            stats_out = gr.Textbox(label="Stats", lines=6, interactive=False)
            video_out = gr.Video(label="Demo video")

    run_btn.click(
        fn=run_training_gradio,
        inputs=[total_timesteps, chunk_steps, log_interval, buffer_size, learning_starts, batch_size, train_freq,
                target_update_interval, tau, learning_rate, gamma, gradient_steps,
                exploration_fraction, exploration_initial_eps, exploration_final_eps,
                verbose, save_after, save_path, record_video],
        outputs=[plot_out, stats_out, video_out],
    )

# In Colab app is shown inline. GPU/CPU auto.
demo.launch(share=False)

**Gradio tip**: Use `demo.launch(share=True)` to get a public URL. On Colab, **GPU runtime** uses CUDA, **CPU runtime** uses CPU automatically.